# Segmentation, one step at a time

Everything the segmenter does lives in **`segment.py`**. This notebook runs it stage by
stage on real flight frames and draws what each stage produced, so a bad pose can be
traced to the stage that caused it instead of guessed at.

There are two paths through the file and this notebook shows both:

| | what it does | where |
|---|---|---|
| **mask path** (§15) | pick a level, sort every pixel, hull what survived, fit an ellipse | `score_channel` → `threshold_mask` → `silhouette_hull` → `fit_ellipse` |
| **direct path** (§16) | build a rim-evidence map, fit the ellipse straight to it | `ring_weight` → `fit_ellipse_image` |

The mask path is now only a **seed** for the direct path. Read
[`theory.md` §15 and §16](theory.md) for why.

Driven per view by `stereo.StereoPoseEstimator._view_candidates`, which is the function
this notebook unrolls.

## 0. Setup

`POSE_APPEARANCE` must be set before `estimator` is imported: `RADIUS_MM` is
bound at import time from `segment.APPEARANCE`.

In [ ]:
import os
# The bench rig as of 2026-08-28 is a white ring on black cloth -> "bright".
# Older flights under results/flights are "dark". Set this to match the take.
os.environ.setdefault("POSE_APPEARANCE", "bright")

import sys, math
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt

POSE = Path.cwd()
if POSE.name != "pose":
    POSE = next(p for p in [POSE / "controller/pose", POSE / "ESP32_PMW/controller/pose"]
                if p.exists())
sys.path[:0] = [str(POSE), str(POSE.parent / "calib"), str(POSE.parent / "camera")]

import conic, segment, estimator, stereo, background as bgmod
import rig as rigmod
from record import open_recording
from shape import CentreCalibration, TiltCalibration

FLIGHTS = POSE.parents[1] / "results" / "flights"
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["image.cmap"] = "gray"

print("appearance   :", segment.APPEARANCE)
print("DARK_THRESH  :", segment.DARK_THRESH, " (level on 255-luminance; larger = stricter)")
print("rim radius   :", estimator.RADIUS_MM, "mm")
print("ring kernel  :", segment.RING_KSIZE, "px   blur", segment.RING_BLUR_SIGMA,
      "  coarse", segment.RING_COARSE_SIGMA)
print("flights      :", [d.name for d in sorted(FLIGHTS.iterdir()) if d.is_dir()])

### Display helpers

One place that decides how things are drawn, so every stage below is one line.

In [ ]:
def show(images, titles=None, cols=None, cmap=None, vlim=None, size=6.0):
    """Draw a row/grid of images with titles. `vlim=(lo, hi)` fixes the scale."""
    images = list(images)
    n = len(images)
    cols = cols or min(n, 3)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows * 0.68))
    axes = np.atleast_1d(axes).ravel()
    for ax, im in zip(axes, images):
        kw = {} if vlim is None else dict(vmin=vlim[0], vmax=vlim[1])
        ax.imshow(im if im.ndim == 3 else im, cmap=cmap, **kw)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis("off")
    if titles:
        for ax, t in zip(axes, titles):
            ax.set_title(t, fontsize=10)
    plt.tight_layout(); plt.show()


def as_bgr(gray):
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)


def draw_ellipse(img, ellipse, colour=(0, 255, 0), thickness=3, label=None):
    """Ellipse drawn on a *copy*, in RGB for matplotlib."""
    out = as_bgr(img) if img.ndim == 2 else img.copy()
    if ellipse is not None:
        (cx, cy), (a, b), ang = ellipse
        cv2.ellipse(out, (int(round(cx)), int(round(cy))),
                    (int(round(a / 2)), int(round(b / 2))), ang, 0, 360, colour, thickness)
        cv2.drawMarker(out, (int(round(cx)), int(round(cy))), colour, cv2.MARKER_CROSS, 30, thickness)
    if label:
        cv2.putText(out, label, (12, 46), cv2.FONT_HERSHEY_SIMPLEX, 1.4, colour, 3)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


def crop_to(img, ellipse, pad=0.45):
    """Zoom on an ellipse -- the rim is a few px thick and full frames hide that."""
    if ellipse is None:
        return img
    (cx, cy), (a, b), _ = ellipse
    r = int(a / 2 * (1 + pad))
    x0, y0 = max(0, int(cx) - r), max(0, int(cy) - r)
    x1, y1 = min(img.shape[1], int(cx) + r), min(img.shape[0], int(cy) + r)
    return img[y0:y1, x0:x1]

## 1. Pick a frame

Change `FLIGHT`, `TAG` and `FRAME` and re-run everything below. §7 finds the bad frames
for you if you do not already know which one is misbehaving.

In [ ]:
FLIGHT = sorted(d for d in FLIGHTS.iterdir() if d.is_dir() and (d / "A" / "A.mp4").exists())[-1]
TAG    = "A"
FRAME  = 600

plates = bgmod.load_for_flight(FLIGHT)

def read_frame(flight, tag, index):
    cap = cv2.VideoCapture(str(flight / tag / f"{tag}.mp4"))
    cap.set(cv2.CAP_PROP_POS_FRAMES, index)
    ok, f = cap.read()
    cap.release()
    if not ok:
        raise IndexError(f"no frame {index} in {tag}")
    return cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) if f.ndim == 3 else f

gray  = read_frame(FLIGHT, TAG, FRAME)
plate = plates[TAG]

print(FLIGHT.name, TAG, FRAME, gray.shape, gray.dtype)
show([gray, plate, cv2.absdiff(gray, plate)],
     ["frame", "empty-rig plate (temporal median)", "|frame - plate|"])

## 2. `valid_region` — where the robot is allowed to be

The dark appearance cannot threshold the whole frame: the room beyond the backdrop is
*darker than the robot* and reaches the border, so an ungated cut hulls the image
(`theory.md` §15.1). Two ways to find the region, and `valid_region` prefers the first:

- `background_mask` — `|frame − plate| > BG_DIFF_THRESH`. 0.056 ms. Everything that
  moved. Note it includes the robot's **shadow**, which is the whole problem.
- `backdrop_mask` — the bright, smooth, convex-hulled region. 2.4 ms. Used only when
  there is no plate.

If this returns `None` the correct answer is *no detection*; see the docstring.

In [ ]:
region_bg   = segment.background_mask(gray, bg=plate)
region_drop = segment.backdrop_mask(gray)
region, source = segment.valid_region(gray, bg=plate, with_source=True)
print("valid_region source:", source,
      "  coverage", f"{100 * (region > 0).mean():.1f}% of frame" if region is not None else "NONE")

show([region_bg, region_drop if region_drop is not None else np.zeros_like(gray),
      cv2.bitwise_and(gray, region) if region is not None else gray],
     ["background_mask (plate diff)", "backdrop_mask (fallback)", "frame inside the region"])

## 3. `score_channel` — one channel where the robot is bright

The only appearance-aware step. For `dark` it returns `bitwise_not(gray) & region` and
the level `DARK_THRESH`, so the cut is on 255 − luminance and a **larger** constant is a
**stricter, darker** cut.

The histogram is the thing to look at: if there is no valley between the rim and the
shadows, no level exists that separates them, and that is exactly the failure §16 was
written for.

In [ ]:
channel, level = segment.score_channel(gray, region=region, background=plate)
print("level:", level, " => the body must read below", 255 - level, "counts")

inside = channel[region > 0] if region is not None else channel.ravel()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.imshow(channel); a1.set_title("score channel (255 - luminance, gated)"); a1.set_xticks([]); a1.set_yticks([])
a2.hist(inside, bins=64, range=(0, 255), log=True)
a2.axvline(level, color="r", lw=2, label=f"DARK_THRESH = {level}")
a2.set_title("histogram inside the region"); a2.set_xlabel("score"); a2.legend()
plt.tight_layout(); plt.show()

## 4. `threshold_mask` — the cut, then morphology

Open 3×3 then close 7×7, and the sizes are not symmetric on purpose: the projected rim
wall is a few pixels thick, so a 5×5 open erodes it away, while the close bridges the
gaps lighting leaves in the ring. Measured at `segment.py:78-86`.

**Watch the mask area.** Over 20 frames per camera it swings 24k → 154k px as shadows
come and go, and the rim is only ~28k of that.

In [ ]:
raw = cv2.threshold(channel, level, 255, cv2.THRESH_BINARY)[1]
opened = cv2.morphologyEx(raw, cv2.MORPH_OPEN, segment._OPEN_KERNEL)
mask = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, segment._CLOSE_KERNEL)
assert np.array_equal(mask, segment.threshold_mask(gray, background=plate, region=region))

for name, m in (("raw cut", raw), ("+ open 3x3", opened), ("+ close 7x7", mask)):
    print(f"{name:14s} {int(m.sum() / 255):7d} px on")
show([raw, opened, mask], ["raw cut", "opened", "closed  <- silhouette_hull sees this"])

## 5. `silhouette_hull` — which blobs are the robot

The rim is **hollow**, so the largest connected blob is the blade cross, not the ring:
measured face-on, largest-contour gives 83 px where the rim is 131. The hull of every
real blob gives 129.9 against 130.7 analytic. That is why this stage exists.

Three rules, in order (`theory.md` §15.5):

1. keep blobs above `_BLOB_KEEP_FRACTION` of the largest,
2. group those within `DARK_MAX_SPREAD` radii of an anchor, then `_regrow` along the
   group's own fitted ellipse — which is what reassembles a rim broken into arcs,
3. among admissible groups (`rms/major ≤ SHAPE_TOL`) take the **largest**, not the
   best-scoring.

In [ ]:
n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, 8)
areas = stats[1:, cv2.CC_STAT_AREA]
keep = 1 + np.flatnonzero(areas >= max(1.0, segment._BLOB_KEEP_FRACTION * areas.max()))
print(f"{n - 1} blobs, {len(keep)} kept above {segment._BLOB_KEEP_FRACTION:.0%} of the largest")
print("kept areas:", sorted(stats[keep, cv2.CC_STAT_AREA], reverse=True)[:12])

chosen = segment._best_group(keep, labels, stats, centroids, n, segment.DARK_MAX_SPREAD)
print("group chosen:", len(chosen), "of", len(keep), "kept blobs")

palette = np.zeros((*mask.shape, 3), np.uint8)
rng = np.random.default_rng(0)
for i in keep:
    palette[labels == i] = rng.integers(60, 255, 3)
picked = np.zeros_like(mask)
for i in chosen:
    picked[labels == i] = 255

hull, area = segment.silhouette_hull(mask, max_spread=segment.DARK_MAX_SPREAD)
hull_img = np.zeros((*mask.shape, 3), np.uint8)
cv2.drawContours(hull_img, [hull.astype(np.int32)], -1, (0, 255, 0), 3)
print(f"hull: {len(hull)} points, {area:.0f} px of blob area")
show([palette, picked, hull_img],
     [f"{len(keep)} kept blobs", f"group chosen ({len(chosen)})", "convex hull"])

## 6. `fit_ellipse` — the mask fit, and its three refinement passes

`fitEllipseDirect` first, then, holding the **angle fixed** throughout (the weights
cannot constrain rotation — letting them try put the axis 33.5° out):

1. **axial** re-weighting, `w = (|proj on major| / a)`, which suppresses the rod and
   magnet sticking out along the rotor axis,
2. **one-sided**, down-weighting points outside the fit — contamination is one-sided
   because the hull is a superset of the rim,
3. **trim**, dropping the 25% most-outward points, but only above 85% angular coverage.

In [ ]:
base = segment.fit_ellipse_direct(hull)
full = segment.fit_ellipse(hull)
plain = segment.fit_ellipse(hull, axial=False)
seg = segment.segment(gray, background=plate)

for name, e in (("fitEllipseDirect", base), ("no axial", plain[0]), ("shipped", full[0])):
    (cx, cy), (a, b), ang = e
    print(f"{name:18s} centre ({cx:7.1f},{cy:7.1f})  axes {a:6.1f} x {b:6.1f}"
          f"  ratio {b / a:.3f}  angle {ang:6.1f}")
print(f"\nsegment(): rms {seg.fit_rms_px:.2f} px  "
      f"rel {seg.fit_rms_px / seg.ellipse[1][0]:.4f}  gate {stereo.MAX_FIT_RMS_REL}"
      f"  -> {'PASS' if seg.fit_rms_px / seg.ellipse[1][0] <= stereo.MAX_FIT_RMS_REL else 'REJECT'}")

over = draw_ellipse(gray, base, (255, 80, 80), 2)
over = cv2.cvtColor(over, cv2.COLOR_RGB2BGR)
cv2.drawContours(over, [hull.astype(np.int32)], -1, (0, 200, 255), 2)
(cx, cy), (a, b), ang = full[0]
cv2.ellipse(over, (int(cx), int(cy)), (int(a / 2), int(b / 2)), ang, 0, 360, (0, 255, 0), 3)
show([cv2.cvtColor(over, cv2.COLOR_BGR2RGB), crop_to(cv2.cvtColor(over, cv2.COLOR_BGR2RGB), full[0])],
     ["hull (orange), direct fit (red), shipped (green)", "zoomed"], cols=2)

### The seed everything below uses

The mask path above is the *legacy* one and on a black backdrop it fails outright -- a
`bright` level of 128 on raw luminance takes the bench, and the hull spans the frame.
That is the finding of §16.10, not a bug to work around, so this cell takes the seed from
whichever path can actually produce one.


In [ ]:
# A seed is plausible if the hull is not spanning the frame.
def plausible(e):
    return e is not None and e[1][0] < 0.6 * min(gray.shape)

if not plausible(seg.ellipse if seg else None):
    print(f"mask path unusable (major {seg.ellipse[1][0]:.0f} px) -- seeding from the map")
    seg, _ = segment.segment_ring(gray)
    assert seg is not None, "no seed from either path"
print(f"seed: major {seg.ellipse[1][0]:.1f} px  ratio {seg.ellipse[1][1] / seg.ellipse[1][0]:.2f}")


## 7. `ring_weight` — the rim evidence map (§16)

The mask above is a threshold's opinion about every pixel. This is the alternative: a
**closing** fills every dark feature narrower than the kernel, so `closing − image` is
non-zero only on dark structures *thinner than the kernel*. The 8 px rim scores in full;
a cast shadow is broader, survives its own closing, and scores ~0.

Subtracting the plate's response removes the static thin dark lines between the coil
formers — the one thing in the scene shaped like the rim.

**This map is never thresholded.** It is a weight.

In [ ]:
k = segment.RING_KSIZE
kern = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))

# Polarity follows APPEARANCE and is the only appearance-aware step in this path.
#   dark   : black-hat, closing - image   (a dark rim on a light ground)
#   bright : top-hat,   image - opening   (a light rim on a dark ground)
DARK = segment.APPEARANCE == "dark"
morph = cv2.MORPH_CLOSE if DARK else cv2.MORPH_OPEN
env = cv2.morphologyEx(gray, morph, kern)
hat = (cv2.subtract(env.astype(np.float32), gray.astype(np.float32)) if DARK
       else cv2.subtract(gray.astype(np.float32), env.astype(np.float32)))
p_env = cv2.morphologyEx(plate, morph, kern)
plate_hat = (cv2.subtract(p_env.astype(np.float32), plate.astype(np.float32)) if DARK
             else cv2.subtract(plate.astype(np.float32), p_env.astype(np.float32)))
W = segment.ring_weight(gray, background=plate)

print(f"appearance {segment.APPEARANCE}: {'closing - image' if DARK else 'image - opening'}"
      f", kernel {k} px (MORPH_RECT: separable, 2.6 ms; MORPH_ELLIPSE would be 40 ms)")
for nm, m in (("hat", hat), ("plate response", plate_hat), ("W = difference, blurred", W)):
    print(f"  {nm:24s} p99 {np.percentile(m, 99):6.1f}   max {m.max():6.1f}")

show([env, hat, plate_hat, W],
     [f"{'closing' if DARK else 'opening'}, {k}x{k}",
      f"{'black' if DARK else 'top'}-hat", "the plate's own response",
      "W (plate subtracted, blurred)"],
     cols=2, vlim=(0, 120))


### The shadow test, made explicit

Pick any dark patch that is *not* the rim and compare. If the rim does not stand well
clear of the shadows here, the kernel is too small for this rim or the plate is stale.

In [ ]:
ring_pts = segment.ellipse_points(seg.ellipse)
on_rim = segment.sample_map(W, ring_pts)

# Everything that is NOT the rim. A plain frame-wide percentile will not do: the ring is
# the brightest thing in W, so it sets its own comparison and the ratio comes out ~1.
elsewhere = np.ones(W.shape, np.uint8)
(ex, ey), (ea, eb), eang = seg.ellipse
cv2.ellipse(elsewhere, (int(ex), int(ey)), (int(ea / 2) + 30, int(eb / 2) + 30),
            eang, 0, 360, 0, -1)
bg_vals = W[elsewhere > 0]

print(f"on the ring       median {np.median(on_rim):6.1f}   p10 {np.percentile(on_rim, 10):6.1f}")
print(f"everything else   median {np.median(bg_vals):6.1f}   p99 {np.percentile(bg_vals, 99):6.1f}"
      f"   max {bg_vals.max():6.1f}")
print(f"\nseparation: the rim's median is "
      f"{np.median(on_rim) / max(np.percentile(bg_vals, 99), 1e-6):.1f}x the brightest"
      f" 1% of everything else")
print("if that is near 1, the kernel is too small for this rim or the plate is stale")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(W.ravel(), bins=80, log=True, label="every pixel")
ax[0].hist(on_rim, bins=40, log=True, alpha=0.8, label="on the fitted ring")
ax[0].legend(); ax[0].set_title("W: rim against everything else"); ax[0].set_xlabel("evidence")
ax[1].plot(np.degrees(np.linspace(0, 2 * np.pi, len(on_rim), endpoint=False)), on_rim)
ax[1].axhline(np.median(on_rim) * segment.RING_COVERAGE_FLOOR, color="r", ls="--",
              label=f"coverage floor ({segment.RING_COVERAGE_FLOOR:g} x median)")
ax[1].set_xlabel("angle around the ring, deg"); ax[1].set_ylabel("evidence")
ax[1].set_title("evidence around the rim -- the dips are shadow and occlusion"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Is the objective actually shaped like a basin?

The direct fit maximises the mean evidence around the predicted ellipse. Before trusting
it, look at the objective: it should have a sharp peak, and the mask fit should be
sitting *off* it — that offset is the bias the direct fit exists to remove.

The fall-off is what sets the **capture radius**, and it is why the coarse pass exists:
a seed further out than this has no gradient to descend.

In [ ]:
def score(e):
    return float(np.mean(segment.sample_map(W, segment.ellipse_points(e))))

(cx, cy), (a, b), ang = seg.ellipse
scales = np.arange(0.80, 1.21, 0.02)
shifts = np.arange(-40, 41, 4)

sc = [score(((cx, cy), (a * s, b * s), ang)) for s in scales]
dx = [score(((cx + d, cy), (a, b), ang)) for d in shifts]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(scales, sc, "o-"); ax[0].axvline(1.0, color="r", ls="--", label="the mask fit")
ax[0].set_xlabel("scale on both axes"); ax[0].set_ylabel("mean evidence"); ax[0].legend()
ax[1].plot(shifts, dx, "o-"); ax[1].axvline(0, color="r", ls="--", label="the mask fit")
ax[1].set_xlabel("centre shift in x, px"); ax[1].legend()
for a_ in ax: a_.grid(alpha=0.3)
fig.suptitle("the objective around the mask fit"); plt.tight_layout(); plt.show()

print(f"peak at scale {scales[int(np.argmax(sc))]:.2f}  and  dx {shifts[int(np.argmax(dx))]:+d} px"
      f"   (1.00 and 0 would mean the mask fit was already right)")

## 9. `fit_ellipse_image` — fitting the ellipse to the image

Coarse pass on a heavily blurred copy of `W` to widen the basin out to the seed, then the
fine pass. `coarse=0` disables it, which is what the tracked path does when the seed is
only one frame old.

`coverage` is the blunder test: the fraction of samples carrying at least half the ring's
**own** median evidence. It is scale-free, so it does not move with the lighting, and a
fit that has slid onto scattered specks keeps a decent mean but loses its coverage.

In [ ]:
coarse_only = segment.fit_ellipse_image(segment._box_blur(W, segment.RING_COARSE_SIGMA),
                                        seg.ellipse, coarse=0.0)
fit = segment.fit_ellipse_image(W, seg.ellipse)

rows = [("mask seed", seg.ellipse, score(seg.ellipse), float("nan")),
        ("after coarse", coarse_only.ellipse, coarse_only.evidence, coarse_only.coverage),
        ("after fine", fit.ellipse, fit.evidence, fit.coverage)]
print(f"{'':14s} {'centre':>18s} {'axes':>16s} {'evidence':>9s} {'coverage':>9s}")
for nm, e, ev, cov in rows:
    (ex, ey), (ea, eb), _ = e
    print(f"{nm:14s} ({ex:7.1f},{ey:7.1f}) {ea:7.1f} x{eb:6.1f} {ev:9.1f} {cov:9.2f}")

gate = stereo.MIN_RING_RIDGE
print(f"\nridge {fit.ridge:.2f}  coverage {fit.coverage:.2f}"
      f"   gate is on ridge >= {gate}  ->  {'PASS' if fit.ridge >= gate else 'REJECT'}")
print(f"centre moved {np.hypot(fit.ellipse[0][0] - cx, fit.ellipse[0][1] - cy):.1f} px,"
      f" major {100 * (fit.ellipse[1][0] / a - 1):+.1f}%")

over = cv2.cvtColor(as_bgr(gray), cv2.COLOR_BGR2RGB)
over = cv2.cvtColor(over, cv2.COLOR_RGB2BGR)
for e, col in ((seg.ellipse, (80, 80, 255)), (fit.ellipse, (0, 255, 0))):
    (ex, ey), (ea, eb), eang = e
    cv2.ellipse(over, (int(ex), int(ey)), (int(ea / 2), int(eb / 2)), eang, 0, 360, col, 3)
over = cv2.cvtColor(over, cv2.COLOR_BGR2RGB)
wv = np.clip(W / max(W.max(), 1e-6) * 255, 0, 255).astype(np.uint8)
show([over, crop_to(over, fit.ellipse), crop_to(draw_ellipse(wv, fit.ellipse), fit.ellipse)],
     ["mask fit (red) vs direct fit (green)", "zoomed", "the direct fit on W"], cols=3)

## 10. Both cameras, the way the estimator sees them

`_view_candidates` runs everything above per view, then back-projects. A view where the
robot is out of frame or fully occluded **should** produce no detection or low coverage —
that is a correct answer, not a failure.

In [ ]:
rig = rigmod.StereoRig.load(rigmod.DEFAULT_PATH)
est = stereo.StereoPoseEstimator(rig, backgrounds=plates,
                                 tilt_cal=TiltCalibration.load(),
                                 centre_cal=CentreCalibration.load())
frames = [read_frame(FLIGHT, t, FRAME) for t in "AB"]
pose = est.update(frames, t=FRAME / 60.0, frame_index=FRAME)

panels, titles = [], []
for f, tag, cam in zip(frames, "AB", rig.cameras):
    s, cands, _ = est._view_candidates(f, cam)
    if s is None:
        panels.append(cv2.cvtColor(as_bgr(f), cv2.COLOR_BGR2RGB)); titles.append(f"{tag}: NO DETECTION")
        continue
    img = draw_ellipse(f, s.ellipse_mask, (80, 80, 255), 2)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    (ex, ey), (ea, eb), eang = s.ellipse
    cv2.ellipse(img, (int(ex), int(ey)), (int(ea / 2), int(eb / 2)), eang, 0, 360, (0, 255, 0), 3)
    panels.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    titles.append(f"{tag}: ridge {s.ridge:.2f}  evidence {s.evidence:.0f}"
                  f"  {len(cands)} candidates")
show(panels, titles, cols=2)

if pose is None:
    print("no stereo pose. counters:",
          {k: v for k, v in vars(est).items() if k.startswith("n_") and isinstance(v, int)})
else:
    print(f"xyz {np.round(pose.xyz_mm, 2)} mm   discrepancy {pose.discrepancy_mm:.2f} mm"
          f"   gate {stereo.MAX_DISCREPANCY_MM:.1f}")

## 11. Which frames are failing?

Rather than guessing at a frame, sweep the take and rank by coverage. Set `FRAME` to one
of the worst and re-run from §1 to see what that stage did.

In [ ]:
STEP = 12

def sweep(flight, tag, step=STEP):
    """Ridge, evidence and seed size for every `step`-th frame of one view."""
    cap = cv2.VideoCapture(str(flight / tag / f"{tag}.mp4"))
    plate, out, i = plates.get(tag), [], -1
    while True:
        i += 1
        ok, f = cap.read()
        if not ok:
            break
        if i % step:
            continue
        g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) if f.ndim == 3 else f
        # The reduced path (S16.10): the seed comes from the map, so this sweep works
        # on a backdrop that has no plate and no valid region.
        s, w = segment.segment_ring(g, background=plate)
        if s is None:
            out.append((i, np.nan, np.nan, 0)); continue
        r = segment.fit_ellipse_image(w, s.ellipse)
        out.append((i, r.ridge, r.evidence, s.ellipse[1][0]) if r
                   else (i, np.nan, np.nan, s.ellipse[1][0]))
    cap.release()
    return np.array(out, dtype=float)

sw = sweep(FLIGHT, TAG)
ok = np.isfinite(sw[:, 1])
gate = stereo.MIN_RING_RIDGE
print(f"{len(sw)} frames sampled   seeded {ok.sum()}   ridge >= {gate}: "
      f"{100 * np.mean(sw[ok, 1] >= gate):.0f}%")

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].semilogy(sw[:, 0], np.maximum(sw[:, 1], 1e-2), ".-")
ax[0].axhline(gate, color="r", ls="--", label="gate")
ax[0].set_ylabel("ridge"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(sw[:, 0], sw[:, 3], ".-", color="tab:orange")
ax[1].set_ylabel("seed major, px"); ax[1].set_xlabel("frame"); ax[1].grid(alpha=0.3)
ax[1].set_title("seed size -- a jump to frame-scale means the hull ran away", fontsize=10)
plt.tight_layout(); plt.show()

worst = sw[ok][np.argsort(sw[ok, 1])][:6]
print("\nworst frames by ridge:")
for i, rg, ev, mj in worst:
    print(f"   FRAME = {int(i):5d}   ridge {rg:6.2f}   evidence {ev:6.1f}   seed major {mj:6.0f} px")


In [ ]:
# The worst six, seed (red) against direct fit (green).
panels, titles = [], []
for i, rg, ev, _ in worst:
    g = read_frame(FLIGHT, TAG, int(i))
    s, w = segment.segment_ring(g, background=plates.get(TAG))
    if s is None:
        continue
    r = segment.fit_ellipse_image(w, s.ellipse)
    img = cv2.cvtColor(draw_ellipse(g, s.ellipse, (80, 80, 255), 2), cv2.COLOR_RGB2BGR)
    (ex, ey), (ea, eb), eang = r.ellipse
    cv2.ellipse(img, (int(ex), int(ey)), (int(ea / 2), int(eb / 2)), eang, 0, 360, (0, 255, 0), 3)
    panels.append(cv2.cvtColor(crop_to(img, r.ellipse, 0.7), cv2.COLOR_BGR2RGB))
    titles.append(f"frame {int(i)}  ridge {rg:.2f}")
show(panels, titles, cols=3, size=4.5)


## 13. The reduced path (§16.10)

With a black backdrop none of §2–§6 is needed: no plate, no `valid_region`, no level on
luminance. The top-hat keeps thin bright structures and the bench, cloth folds and room
are all broad, so the map finds the robot on its own and `ring_seed` takes the seed
straight from it. This is what `StereoPoseEstimator(direct=True)` actually runs.


In [ ]:
seg2, W2 = segment.segment_ring(gray)
assert seg2 is not None, "no seed from the evidence map"
fit2 = segment.fit_ellipse_image(W2, seg2.ellipse)

print(f"seed  major {seg2.ellipse[1][0]:6.1f}  ratio {seg2.ellipse[1][1] / seg2.ellipse[1][0]:.2f}")
print(f"fit   major {fit2.ellipse[1][0]:6.1f}  ratio {fit2.ellipse[1][1] / fit2.ellipse[1][0]:.2f}"
      f"   ridge {fit2.ridge:6.2f}  (gate {stereo.MIN_RING_RIDGE})")

vis = np.clip(W2 / max(W2.max(), 1e-6) * 255, 0, 255).astype(np.uint8)
show([vis, seg2.mask, draw_ellipse(gray, fit2.ellipse)],
     ["ring_weight -- no plate, no region", "ring_seed's mask", "the direct fit"])


## 12. Knobs, and what each one breaks

| constant | where | what moving it does |
|---|---|---|
| `DARK_THRESH` | `segment.py` | the mask **seed** only, and unused on the black backdrop -- `ring_seed` replaces it |
| `RING_SEED_FRACTION` | `segment.py` | the seed level, as a fraction of the map's own p99.9. 0.30 |
| `BG_DIFF_THRESH` | `segment.py` | how much a pixel must differ from the plate. Too low admits noise, too high loses a slow-moving robot |
| `DARK_MAX_SPREAD` | `segment.py` | how far a blob may sit from the anchor. Tightest margin of any constant here (passes 1.0–1.7) |
| `RING_KSIZE` | `segment.py` | must exceed the rim's thickness and stay under the shadows' width. 41 |
| `RING_COARSE_SIGMA` | `segment.py` | the capture radius. Too small and a bad seed never recovers |
| `MIN_RING_RIDGE` | `stereo.py` | how far the fit must stand above its surroundings. 2.5 |
| `RADIUS_BY_APPEARANCE` / `RADIUS_BENCH_MM` | `estimator.py` | 10.2 for the dark rig, **10.05** for the white-ring bench rig. Belongs to the *direct* fit; re-run `fit_radius.py` if its constants move |

If the plate is stale — the rig moved, the lighting changed, or the robot hovered in one
spot while it was built — every stage above degrades at once and none of them says so.
`background.check()` is the test; §2's `|frame − plate|` panel is the quick look.